## Silver Layer Transformations

Transform Bronze data into clean, standardized, and analysis-ready Delta tables.

**Pipeline Flow**

Bronze Delta Tables  
↓  
Data Cleaning  
↓  
Standardization  
↓  
Business Rule Transformations  
↓  
Data Quality Validation  
↓  
**Silver Delta Tables**

### 1. Configuration

Define the source Bronze layer and target Silver layer locations.

In [0]:
catalog = "workspace"

bronze_schema = "consumer_analytics_bronze"
silver_schema = "consumer_analytics_silver"

table_names = [
    "transactions",
    "products",
    "demographics",
    "campaign_desc",
    "campaign_households",
    "coupons",
    "coupon_redemptions"
]

### 2. Create Silver Schema

Create a dedicated schema for cleaned and standardized Silver Delta tables.

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}
""")

DataFrame[]

In [0]:
spark.sql(
    f"SHOW SCHEMAS IN {catalog}"
).show(truncate=False)

+-------------------------+
|databaseName             |
+-------------------------+
|consumer_analytics_bronze|
|consumer_analytics_silver|
|default                  |
|information_schema       |
+-------------------------+



### 3. Load Bronze Delta Tables

Load the Bronze Delta tables from Unity Catalog as the source for Silver transformations.

In [0]:
bronze_dfs = {}

for table_name in table_names:

    source_table = f"{catalog}.{bronze_schema}.{table_name}"

    bronze_dfs[table_name] = spark.table(source_table)

    print(f"{source_table}: loaded")

workspace.consumer_analytics_bronze.transactions: loaded
workspace.consumer_analytics_bronze.products: loaded
workspace.consumer_analytics_bronze.demographics: loaded
workspace.consumer_analytics_bronze.campaign_desc: loaded
workspace.consumer_analytics_bronze.campaign_households: loaded
workspace.consumer_analytics_bronze.coupons: loaded
workspace.consumer_analytics_bronze.coupon_redemptions: loaded


### 4. Validate Bronze Inputs

Validate that all expected Bronze tables are available before applying Silver transformations.

In [0]:
for table_name, df in bronze_dfs.items():
    print(f"{table_name}: {df.count():,} rows")

transactions: 2,595,732 rows
products: 92,353 rows
demographics: 801 rows
campaign_desc: 30 rows
campaign_households: 7,208 rows
coupons: 124,548 rows
coupon_redemptions: 2,318 rows


### 5. Initialize Silver DataFrames

Create Silver transformation DataFrames from the Bronze inputs while preserving the original Bronze tables unchanged.

In [0]:
silver_dfs = {
    table_name: df
    for table_name, df in bronze_dfs.items()
}

### 6. Deduplicate Coupon Data

Source profiling identified redundant duplicate records in the coupon dataset.

The expected business grain is:

**COUPON_UPC + PRODUCT_ID + CAMPAIGN**

Duplicate records at this grain are removed in the Silver layer while the original source records remain preserved in Bronze.

In [0]:
coupon_grain = [
    "COUPON_UPC",
    "PRODUCT_ID",
    "CAMPAIGN"
]

silver_dfs["coupons"] = (
    bronze_dfs["coupons"]
    .dropDuplicates(coupon_grain)
)

### 7. Validate Coupon Deduplication

Validate the coupon deduplication by comparing Bronze and Silver row counts.

In [0]:
bronze_coupon_count = bronze_dfs["coupons"].count()
silver_coupon_count = silver_dfs["coupons"].count()

removed_duplicates = bronze_coupon_count - silver_coupon_count

print(f"Bronze coupon rows: {bronze_coupon_count:,}")
print(f"Silver coupon rows: {silver_coupon_count:,}")
print(f"Duplicates removed: {removed_duplicates:,}")

Bronze coupon rows: 124,548
Silver coupon rows: 119,384
Duplicates removed: 5,164


### 8. Standardize Transaction Time

Convert the source `TRANS_TIME` HHMM representation into analysis-friendly hour and minute fields.

The original `TRANS_TIME` column is retained for traceability.

In [0]:
from pyspark.sql import functions as F

silver_dfs["transactions"] = (
    bronze_dfs["transactions"]
    .withColumn(
        "transaction_hour",
        F.floor(F.col("TRANS_TIME") / 100)
    )
    .withColumn(
        "transaction_minute",
        F.col("TRANS_TIME") % 100
    )
)

### 9. Validate Transaction Time

Validate that the derived transaction hour and minute fields fall within valid clock-time ranges.

In [0]:
invalid_time_count = (
    silver_dfs["transactions"]
    .filter(
        (F.col("transaction_hour") < 0) |
        (F.col("transaction_hour") > 23) |
        (F.col("transaction_minute") < 0) |
        (F.col("transaction_minute") > 59)
    )
    .count()
)

print(f"Invalid transaction times: {invalid_time_count:,}")

Invalid transaction times: 0


In [0]:
display(
    silver_dfs["transactions"]
    .filter(F.col("TRANS_TIME").isin(5, 30, 100, 115, 930, 1425, 2359))
    .select(
        "TRANS_TIME",
        "transaction_hour",
        "transaction_minute"
    )
    .distinct()
    .orderBy("TRANS_TIME")
)

TRANS_TIME,transaction_hour,transaction_minute
5,0,5
30,0,30
100,1,0
115,1,15
930,9,30
1425,14,25
2359,23,59


### 10. Standardize Categorical Text Fields

Standardize categorical text fields by removing leading and trailing whitespace.

This improves consistency for downstream grouping, filtering, joins, and reporting while preserving the original category values.

In [0]:
product_text_columns = [
    "DEPARTMENT",
    "BRAND",
    "COMMODITY_DESC",
    "SUB_COMMODITY_DESC",
    "CURR_SIZE_OF_PRODUCT"
]

demographic_text_columns = [
    "AGE_DESC",
    "MARITAL_STATUS_CODE",
    "INCOME_DESC",
    "HOMEOWNER_DESC",
    "HH_COMP_DESC",
    "HOUSEHOLD_SIZE_DESC",
    "KID_CATEGORY_DESC"
]

for column_name in product_text_columns:
    silver_dfs["products"] = (
        silver_dfs["products"]
        .withColumn(column_name, F.trim(F.col(column_name)))
    )

for column_name in demographic_text_columns:
    silver_dfs["demographics"] = (
        silver_dfs["demographics"]
        .withColumn(column_name, F.trim(F.col(column_name)))
    )

### 11. Validate Text Standardization

Check whether leading or trailing whitespace remains in the standardized categorical fields.

In [0]:
for column_name in product_text_columns:

    remaining_whitespace = (
        silver_dfs["products"]
        .filter(
            F.col(column_name) != F.trim(F.col(column_name))
        )
        .count()
    )

    print(f"products.{column_name}: {remaining_whitespace:,}")

products.DEPARTMENT: 0
products.BRAND: 0
products.COMMODITY_DESC: 0
products.SUB_COMMODITY_DESC: 0
products.CURR_SIZE_OF_PRODUCT: 0


### 12. Add Silver Processing Metadata

Add a Silver processing timestamp while preserving the Bronze ingestion metadata for end-to-end lineage.

In [0]:
for table_name in silver_dfs:

    silver_dfs[table_name] = (
        silver_dfs[table_name]
        .withColumn("_processed_at", F.current_timestamp())
    )

In [0]:
for table_name in silver_dfs:

    silver_dfs[table_name] = (
        silver_dfs[table_name]
        .withColumn("_processed_at", F.current_timestamp())
    )

### 13. Write Silver Delta Tables

Persist the cleaned and standardized DataFrames as managed Delta tables in the Silver schema.

In [0]:
for table_name, df in silver_dfs.items():

    target_table = f"{catalog}.{silver_schema}.{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    print(f"{target_table}: written successfully")

workspace.consumer_analytics_silver.transactions: written successfully
workspace.consumer_analytics_silver.products: written successfully
workspace.consumer_analytics_silver.demographics: written successfully
workspace.consumer_analytics_silver.campaign_desc: written successfully
workspace.consumer_analytics_silver.campaign_households: written successfully
workspace.consumer_analytics_silver.coupons: written successfully
workspace.consumer_analytics_silver.coupon_redemptions: written successfully


### 14. Validate Silver Tables

Verify that all expected Silver Delta tables were successfully created in Unity Catalog.

In [0]:
display(
    spark.sql(
        f"SHOW TABLES IN {catalog}.{silver_schema}"
    )
)

database,tableName,isTemporary
consumer_analytics_silver,campaign_desc,false
consumer_analytics_silver,campaign_households,false
consumer_analytics_silver,coupon_redemptions,false
consumer_analytics_silver,coupons,false
consumer_analytics_silver,demographics,false
consumer_analytics_silver,products,false
consumer_analytics_silver,transactions,false


### 15. Validate Silver Row Counts

Compare Bronze and Silver row counts to confirm expected transformations and identify any unintended record loss.

In [0]:
for table_name in table_names:

    bronze_count = bronze_dfs[table_name].count()
    silver_count = spark.table(
        f"{catalog}.{silver_schema}.{table_name}"
    ).count()

    difference = bronze_count - silver_count

    print(
        f"{table_name}: "
        f"bronze={bronze_count:,} | "
        f"silver={silver_count:,} | "
        f"difference={difference:,}"
    )

transactions: bronze=2,595,732 | silver=2,595,732 | difference=0
products: bronze=92,353 | silver=92,353 | difference=0
demographics: bronze=801 | silver=801 | difference=0
campaign_desc: bronze=30 | silver=30 | difference=0
campaign_households: bronze=7,208 | silver=7,208 | difference=0
coupons: bronze=124,548 | silver=119,384 | difference=5,164
coupon_redemptions: bronze=2,318 | silver=2,318 | difference=0
